# 60 Summary over transects

<!-- navigation -->
[<kbd>← 52 Timflow model</kbd>](52%20Timflow%20model.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md)

**Optional**: after notebook 52 for two or more transects

Combines the model summaries and response factors of all transects into one table and figures.

| | |
|---|---|
| **Input** | `model_summary.xlsx`, `responsefactors.xlsx` and `responsefactors_x.xlsx` in the 'results_final' folder of each transect (notebook 52) |
| **Output** | `summary_transects.xlsx`, `responsefactors_transects.xlsx` and figures in the project folder 'results' |

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt

import matplotlib.pyplot as plt
import pandas as pd
tlt.show_versions()

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

In [ ]:
transects = project.transects()  # or a selection, e.g. ["Papendrecht West", "Papendrecht East"]

## 1. Collect the results

In [ ]:
summaries, rf_tables, rf_profiles = [], [], {}
for transect in transects:
    folder = project.transect_dir(transect, "final")
    fn_summary, fn_rf = folder / "model_summary.xlsx", folder / "responsefactors.xlsx"
    if not (fn_summary.exists() and fn_rf.exists()):
        print(f"{transect}: no results yet (run notebook 52 for this transect)")
        continue
    summaries.append(pd.read_excel(fn_summary, index_col=0))
    rf = pd.read_excel(fn_rf, index_col=0)
    rf.insert(0, "transect", transect)
    rf_tables.append(rf)
    # Response factors along x (lines in section 3); missing in results of older versions of notebook 52
    fn_profiles = folder / "responsefactors_x.xlsx"
    if fn_profiles.exists():
        rf_profiles[transect] = pd.read_excel(fn_profiles)
    else:
        print(f"{transect}: no responsefactors_x.xlsx (run notebook 52 again for the lines in section 3)")

summary = pd.concat(summaries)
# Ratio of the leakage factors riverside / landside
summary["leakage_factor_ratio"] = summary["leakage_factor_riverside_m"] / summary["leakage_factor_landside_m"]
responsefactors = pd.concat(rf_tables)

project.results_dir.mkdir(parents=True, exist_ok=True)
tlt.export_to_excel(summary, project.results_dir / "summary_transects.xlsx")
rf_columns = [c for c in responsefactors.columns if c.startswith("rf_")]
lag_columns = [c for c in responsefactors.columns if c.startswith("lag_")]
tlt.export_to_excel(
    responsefactors,
    project.results_dir / "responsefactors_transects.xlsx",
    number_formats={**{c: "0.0%" for c in rf_columns}, **{c: "0.0" for c in lag_columns}},
)

summary.T.style.format(precision=3)

## 2. Leakage factors per transect
Leakage factors of the calibrated steady model: riverside (river zone) and landside (hinterland), and their ratio
riverside / landside.

With more than one aquifer, a zone has as many leakage factors as aquifers: Timflow gives them in `lab`, sorted from
large to small (the eigenvalues of the system of aquifers, not one value per layer). The value shown is `lab[0]`, the
**largest** leakage factor, not that of the uppermost layer: it dominates the response at larger distances from the river.
See notebook 52, section 4.1.

In [ ]:
columns = {
    "riverside": "leakage_factor_riverside_m",
    "landside": "leakage_factor_landside_m",
}
f, (ax, ax_ratio) = plt.subplots(1, 2, figsize=(12, 5), layout="constrained", width_ratios=(2, 1))
summary[list(columns.values())].rename(columns={v: k for k, v in columns.items()}).plot.bar(ax=ax, rot=0)
ax.set_ylabel(f"leakage factor [{units['x']}]")
ax.set_xlabel("")
ax.legend(loc=(0, 1), frameon=False, ncol=4, fontsize="x-small")
ax.grid(True, axis="y")

summary["leakage_factor_ratio"].plot.bar(ax=ax_ratio, rot=0, color="C2")
ax_ratio.set_ylabel(r"ratio $\lambda$ riverside / $\lambda$ landside [-]")
ax_ratio.set_xlabel("")
ax_ratio.grid(True, axis="y")
tlt.finalize_and_save(f, project.results_dir, project.name, tag="leakage_factors", file_label="")

## 3. Response factors along the cross-section
Modelled response factors per transect: one row per model layer, one column per method. Lines along x (calculated in
notebook 52 at `n_x_profile` points over the x range of its cross-section figures), markers at the piezometers in that
layer. From left to right: steady, design hydrograph (MHW) and tidal, from slow to fast changes of the river level. Set
`xlim` to choose the x range; outside the range of notebook 52 there are only markers.

In [ ]:
## UPDATE FOR YOUR PROJECT
xlim = None  # x range [m] of the figure, e.g. (-100, 250); None = the full range

methods = [c for c in ("rf_steady_mod", "rf_mhw_mod", "rf_tidal_mod") if c in responsefactors.columns]
layers = sorted(
    set(responsefactors["modellayer"].astype(int))
    | {int(layer) for profile in rf_profiles.values() for layer in profile["layer"].unique()}
)
f, axes = plt.subplots(
    len(layers), len(methods), figsize=(5 * len(methods), 3 * len(layers)),
    sharex=True, sharey=True, layout="constrained", squeeze=False,
)
for row, layer in enumerate(layers):
    for col, method in enumerate(methods):
        ax = axes[row, col]
        for i, (transect, group) in enumerate(responsefactors.groupby("transect", sort=False)):
            color = f"C{i}"
            points = group[group["modellayer"] == layer].sort_values("x_crosssection")
            profile = rf_profiles.get(transect)
            if profile is not None and method in profile and layer in set(profile["layer"]):
                line = profile[profile["layer"] == layer]
                ax.plot(line["x [m]"], line[method], "-", color=color, label=transect)
                ax.plot(points["x_crosssection"], points[method], "o", color=color)
            elif not points.empty:  # no profile: markers at the piezometers, connected
                ax.plot(points["x_crosssection"], points[method], "o-", color=color, label=transect)
        if row == 0:
            ax.set_title(method, loc="right")
        if col == 0:
            ax.set_ylabel(f"layer {layer}\nresponse factor [-]")
        if row == len(layers) - 1:
            ax.set_xlabel(f"x [{units['x']}]")
        if xlim is not None:
            ax.set_xlim(xlim)
        ax.grid(True)
axes[0, 0].legend(loc=(0, 1), frameon=False, ncol=4, fontsize="x-small")
tlt.finalize_and_save(f, project.results_dir, project.name, tag="responsefactors_transects", file_label="")

---

<!-- navigation -->
[<kbd>← 52 Timflow model</kbd>](52%20Timflow%20model.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md)